In [1]:
import numpy as np
import torch
import lightning.pytorch as pl
import torchmetrics
import torchvision
from torchinfo import summary

class NeuralNetwork(pl.LightningModule):
    def __init__(self,
                 input_size: int,
                 output_size: int,
                 **kwargs):
        
        super().__init__(**kwargs)
        size = 32
        self.conv1 = torch.nn.Conv2d(3,   16,   5, 1, 0)
        self.conv2 = torch.nn.Conv2d(16,  32,   5, 1, 0)
        self.pool1_2 = torch.nn.MaxPool2d(2)
        self.conv_activation = torch.nn.ReLU()
        
        self.flatten_layer = torch.nn.Flatten()
        self.output_layer = torch.nn.Linear(4608,output_size)
        self.output_activation = torch.nn.Softmax(dim=-1)
        self.accuracy = torchmetrics.classification.Accuracy(task='multiclass', num_classes=output_size)
        self.cce_loss = torch.nn.CrossEntropyLoss()

    def forward(self, x):
        y = x
        y = self.conv1(y)
        y = self.conv_activation(y)
        y = self.conv2(y)
        y = self.conv_activation(y)
        y = self.pool1_2(y)
        y = self.flatten_layer(y)
        y = self.output_layer(y)
        return y

    # Note - not a PyTorch Lightning Function
    def predict(self, x):
        y = self.forward(x)
        y = self.output_activation(y)
        return y

    def configure_optimizers(self):
       optimizer = torch.optim.Adam(self.parameters(), lr=1e-5)
       return optimizer

    def predict_step(self, predict_batch, batch_idx):
        x, y = predict_batch
        y_pred = self.predict(x)
        return y_pred, y_true

    def training_step(self, train_batch, batch_idx):
        x, y = train_batch
        y_pred = self.forward(x)
        loss = self.cce_loss(y_pred, y)
        acc = self.accuracy(y_pred, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True)
        self.log('train_acc', acc, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch
        y_pred = self.forward(x)
        loss = self.cce_loss(y_pred,y)
        acc = self.accuracy(y_pred, y)
        self.log('val_loss', loss, on_step=False, on_epoch=True)
        self.log('val_acc', acc, on_step=False, on_epoch=True)
        return loss

    def test_step(self, test_batch, batch_idx):
        x, y = test_batch
        y_pred = self.forward(x)
        loss = self.cce_loss(y_pred,y)
        acc = self.accuracy(y_pred, y)
        self.log('test_loss', loss, on_step=False, on_epoch=True)
        self.log('test_acc', acc, on_step=False, on_epoch=True)
        return loss

In [2]:
class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(self,
                 batch_size = 128,
                 val_split = 0.2,
                 num_workers = 4,
                 location = "~/datasets",
                 **kwargs):
        super().__init__(**kwargs)
        self.batch_size = batch_size
        self.val_split = val_split
        self.num_workers = num_workers
        self.location = location
        self.input_shape = (3, 32, 32)
        self.output_shape = 10
        self.data_train = None
        self.data_val = None
        self.data_test = None

    def setup(self, stage: str):
        if (stage == 'fit' or \
            stage == 'validate') and \
            not(self.data_train and self.data_val):
            training_dataset = torchvision.datasets.CIFAR10(root=self.location,download=True, train=True)
            x_train = training_dataset.data.transpose(0, 3, 2, 1)
            # x_train = training_dataset.data.reshape(training_dataset.data.shape[0:1] + (-1,)) # seems to flatten
            x_train = (x_train / 127.5) - 1.0
            y_train = np.array(training_dataset.targets)
            rng = np.random.default_rng(0)
            permutation = rng.permutation(x_train.shape[0])
            split_point = int(x_train.shape[0]*(1.0-self.val_split))
            self.data_train = list(zip(torch.Tensor(x_train[permutation[:split_point]]).to(torch.float32),
                                       torch.Tensor(y_train[permutation[:split_point]]).to(torch.long)))
            self.data_val = list(zip(torch.Tensor(x_train[permutation[split_point:]]).to(torch.float32),
                                     torch.Tensor(y_train[permutation[split_point:]]).to(torch.long)))
        if (stage == 'test' or \
            stage == 'predict') and \
            not self.data_test:
            testing_dataset = torchvision.datasets.CIFAR10(root=self.location,download=True, train=False)
            x_test = testing_dataset.data.transpose(0, 3, 2, 1)
            x_test = (x_test / 127.5) - 1.0
            y_test = np.array(testing_dataset.targets)
            self.data_test = list(zip(torch.Tensor(x_test).to(torch.float32),
                                      torch.Tensor(y_test).to(torch.long)))
            
    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.data_train,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.data_val,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=False)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.data_test,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=False)

    def predict_dataloader(self):
        return torch.utils.data.DataLoader(self.data_test,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=False)

In [3]:
def train_and_log_and_test(trainer, model, data_module):
    # validate
    result = trainer.validate(model, dataloaders=data_module)
    # train
    result = trainer.fit(model, data_module)
    # validate again
    result = trainer.validate(model, data_module)
    # test
    result = trainer.test(model, data_module)

EPOCHS = 1

cifar10 = CIFAR10DataModule()
model = NeuralNetwork((1, 3, 32, 32), 10)
print(summary(model, input_size=(1, 3, 32, 32)))

logger = pl.loggers.CSVLogger("logs",name="cifar10-conv2d-bottleneck",)
trainer = pl.Trainer(max_epochs=EPOCHS,
                     logger=logger,
                     enable_progress_bar=True,
                     log_every_n_steps=0,
                     callbacks=[pl.callbacks.TQDMProgressBar(refresh_rate=20)])

train_and_log_and_test(trainer, model, cifar10)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Layer (type:depth-idx)                   Output Shape              Param #
NeuralNetwork                            [1, 10]                   --
├─Conv2d: 1-1                            [1, 16, 28, 28]           1,216
├─ReLU: 1-2                              [1, 16, 28, 28]           --
├─Conv2d: 1-3                            [1, 32, 24, 24]           12,832
├─ReLU: 1-4                              [1, 32, 24, 24]           --
├─MaxPool2d: 1-5                         [1, 32, 12, 12]           --
├─Flatten: 1-6                           [1, 4608]                 --
├─Linear: 1-7                            [1, 10]                   46,090
Total params: 60,138
Trainable params: 60,138
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 8.39
Input size (MB): 0.01
Forward/backward pass size (MB): 0.25
Params size (MB): 0.24
Estimated Total Size (MB): 0.50


100%|███████████████████████████| 170M/170M [04:25<00:00, 642kB/s]
/home/adrianm/Documents/projects/vinyl-record-cleaner/.venv/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Validation DataLoader 0: 100%|████| 79/79 [00:01<00:00, 56.86it/s]


  | Name              | Type               | Params | Mode  | FLOPs
-------------------------------------------------------------------------
0 | conv1             | Conv2d             | 1.2 K  | train | 0    
1 | conv2             | Conv2d             | 12.8 K | train | 0    
2 | pool1_2           | MaxPool2d          | 0      | train | 0    
3 | conv_activation   | ReLU               | 0      | train | 0    
4 | flatten_layer     | Flatten            | 0      | train | 0    
5 | output_layer      | Linear             | 46.1 K | train | 0    
6 | output_activation | Softmax            | 0      | train | 0    
7 | accuracy          | MulticlassAccuracy | 0      | train | 0    
8 | cce_loss          | CrossEntropyLoss   | 0      | train | 0    
-------------------------------------------------------------------------
60.1 K    Trainable params
0         Non-trainable params
60.1 K    Total params
0.241     Total estimated model params size (MB)
9         Modules in train mode
0        


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.10189999639987946
        val_loss             2.301410436630249
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Epoch 0: 100%|█████████| 313/313 [00:25<00:00, 12.16it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=1` reached.


Validation DataLoader 0: 100%|████| 79/79 [00:01<00:00, 40.87it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc             0.259799987077713
        val_loss            2.1787145137786865
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Testing DataLoader 0: 100%|███████| 79/79 [00:02<00:00, 37.27it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.25999999046325684
        test_loss           2.1